# Lab

- Use this lab to practice the chapter's coding skills in runnable Python code.
- Sign in from the account menu before you start working on the lab.
- Launch Live Code (the rocket) to write and run code in the coding cells.
- Multiple submissions are allowed. The highest score is kept.

<div data-lab-answers-release-at="2026-12-10T23:59-06:00"></div>


In [ ]:
### Question 1: The Assignment Step
# Six customers (in standard units) and two current cluster centers are below.
#   1. Assign each customer to the nearer center (0 or 1) using distance.
#   2. Print the assignments as a list and the number of customers in each cluster.
# Print exactly:
# Assignments: [0, 0, 1, 1, 0, 1]
# Cluster sizes: [3, 3]
import numpy as np
import pandas as pd

points = pd.DataFrame({
    "spend_su": [-1.0, -0.8, 1.1, 0.9, -1.2, 1.3],
    "orders_su": [-0.5, -0.9, 0.8, 1.2, -0.7, 0.6],
})
centers = np.array([[-1.0, -1.0], [1.0, 1.0]])
### Your code starts here.



### Your code ends here.


In [1]:
### Solution
import numpy as np
import pandas as pd

points = pd.DataFrame({
    "spend_su": [-1.0, -0.8, 1.1, 0.9, -1.2, 1.3],
    "orders_su": [-0.5, -0.9, 0.8, 1.2, -0.7, 0.6],
})
centers = np.array([[-1.0, -1.0], [1.0, 1.0]])

assignments = []
for spend, orders in zip(points['spend_su'], points['orders_su']):
    point = np.array([spend, orders])
    d0 = np.sqrt(np.sum((point - centers[0]) ** 2))
    d1 = np.sqrt(np.sum((point - centers[1]) ** 2))
    if d0 <= d1:
        assignments.append(0)
    else:
        assignments.append(1)

print("Assignments:", assignments)
print("Cluster sizes:", [assignments.count(0), assignments.count(1)])


Assignments: [0, 0, 1, 1, 0, 1]
Cluster sizes: [3, 3]


In [ ]:
### Question 2: The Update Step
# Using the assignments from Question 1 (given below), move each center to the
# mean of the customers assigned to it.
#   1. Use groupby on the cluster column to compute the new centers.
#   2. Print each new center rounded to 2 decimals, cluster 0 first.
# Print exactly:
# Center 0: [-1.0, -0.7]
# Center 1: [1.1, 0.87]
import numpy as np
import pandas as pd

points = pd.DataFrame({
    "spend_su": [-1.0, -0.8, 1.1, 0.9, -1.2, 1.3],
    "orders_su": [-0.5, -0.9, 0.8, 1.2, -0.7, 0.6],
    "cluster": [0, 0, 1, 1, 0, 1],
})
### Your code starts here.



### Your code ends here.


In [2]:
### Solution
import numpy as np
import pandas as pd

points = pd.DataFrame({
    "spend_su": [-1.0, -0.8, 1.1, 0.9, -1.2, 1.3],
    "orders_su": [-0.5, -0.9, 0.8, 1.2, -0.7, 0.6],
    "cluster": [0, 0, 1, 1, 0, 1],
})

new_centers = points.groupby('cluster').mean().round(2)
print("Center 0:", new_centers.loc[0].tolist())
print("Center 1:", new_centers.loc[1].tolist())


Center 0: [-1.0, -0.7]
Center 1: [1.1, 0.87]


In [ ]:
### Question 3: Raw vs. Scaled Features
# Spend is in dollars and orders is a count, so spend dominates the distances.
#   1. Fit KMeans(n_clusters=4, n_init=10, random_state=0) to the raw customers
#      and, separately, to the features in standard units (scaled).
#   2. Print the cluster sizes for each, sorted from smallest to largest.
# Print exactly:
# Raw sizes: [3, 3, 5, 5]
# Scaled sizes: [4, 4, 4, 4]
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

customers = pd.DataFrame({
    "spend": [300, 650, 450, 800, 500, 900, 700, 1100, 2000, 2500, 2900, 2300, 2100, 2700, 3000, 2400],
    "orders": [3, 2, 4, 3, 24, 22, 26, 25, 5, 4, 6, 5, 28, 26, 30, 27],
})
scaled = (customers - customers.mean()) / customers.std(ddof=0)
### Your code starts here.



### Your code ends here.


In [3]:
### Solution
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

customers = pd.DataFrame({
    "spend": [300, 650, 450, 800, 500, 900, 700, 1100, 2000, 2500, 2900, 2300, 2100, 2700, 3000, 2400],
    "orders": [3, 2, 4, 3, 24, 22, 26, 25, 5, 4, 6, 5, 28, 26, 30, 27],
})
scaled = (customers - customers.mean()) / customers.std(ddof=0)

raw_labels = KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(customers)
scaled_labels = KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(scaled)

print("Raw sizes:", sorted(pd.Series(raw_labels).value_counts().tolist()))
print("Scaled sizes:", sorted(pd.Series(scaled_labels).value_counts().tolist()))
# Without scaling, k-means splits customers mostly by spend and mixes frequent
# and occasional buyers. With scaling, it finds the four customer types.


Raw sizes: [3, 3, 5, 5]
Scaled sizes: [4, 4, 4, 4]


In [ ]:
### Question 4: Choosing k With the Silhouette Score
#   1. For k = 2, 3, 4, and 5, fit KMeans(n_clusters=k, n_init=10, random_state=0)
#      to scaled and compute silhouette_score(scaled, labels), rounded to 3 decimals.
#   2. Print each score, then the k with the highest score.
# Print exactly:
# k=2: 0.457
# k=3: 0.573
# k=4: 0.746
# k=5: 0.652
# Best k: 4
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

customers = pd.DataFrame({
    "spend": [300, 650, 450, 800, 500, 900, 700, 1100, 2000, 2500, 2900, 2300, 2100, 2700, 3000, 2400],
    "orders": [3, 2, 4, 3, 24, 22, 26, 25, 5, 4, 6, 5, 28, 26, 30, 27],
})
scaled = (customers - customers.mean()) / customers.std(ddof=0)
### Your code starts here.



### Your code ends here.


In [4]:
### Solution
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

customers = pd.DataFrame({
    "spend": [300, 650, 450, 800, 500, 900, 700, 1100, 2000, 2500, 2900, 2300, 2100, 2700, 3000, 2400],
    "orders": [3, 2, 4, 3, 24, 22, 26, 25, 5, 4, 6, 5, 28, 26, 30, 27],
})
scaled = (customers - customers.mean()) / customers.std(ddof=0)

best_k = None
best_score = -1
for k in [2, 3, 4, 5]:
    labels = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(scaled)
    score = round(silhouette_score(scaled, labels), 3)
    print("k=" + str(k) + ":", score)
    if score > best_score:
        best_k = k
        best_score = score
print("Best k:", best_k)


k=2: 0.457
k=3: 0.573
k=4: 0.746
k=5: 0.652
Best k: 4


In [ ]:
### Question 5: Describing the Segments
#   1. Fit KMeans(n_clusters=4, n_init=10, random_state=0) to scaled and add the
#      labels to customers as a column named cluster.
#   2. Use groupby to compute each cluster's mean spend and mean orders in the
#      ORIGINAL units, and sort the clusters by mean spend.
#   3. Print one line per cluster, lowest spend first, with both means rounded
#      to 2 decimals.
# Print exactly:
# Spend 550.0, orders 3.0
# Spend 800.0, orders 24.25
# Spend 2425.0, orders 5.0
# Spend 2550.0, orders 27.75
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

customers = pd.DataFrame({
    "spend": [300, 650, 450, 800, 500, 900, 700, 1100, 2000, 2500, 2900, 2300, 2100, 2700, 3000, 2400],
    "orders": [3, 2, 4, 3, 24, 22, 26, 25, 5, 4, 6, 5, 28, 26, 30, 27],
})
scaled = (customers - customers.mean()) / customers.std(ddof=0)
### Your code starts here.



### Your code ends here.


In [5]:
### Solution
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

customers = pd.DataFrame({
    "spend": [300, 650, 450, 800, 500, 900, 700, 1100, 2000, 2500, 2900, 2300, 2100, 2700, 3000, 2400],
    "orders": [3, 2, 4, 3, 24, 22, 26, 25, 5, 4, 6, 5, 28, 26, 30, 27],
})
scaled = (customers - customers.mean()) / customers.std(ddof=0)

customers['cluster'] = KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(scaled)
profile = customers.groupby('cluster')[['spend', 'orders']].mean().sort_values('spend')

for spend, orders in zip(profile['spend'], profile['orders']):
    print("Spend " + str(round(spend, 2)) + ", orders " + str(round(orders, 2)))
# Occasional buyers, frequent small baskets, big-ticket buyers, and loyal
# high-value customers. The cluster numbers themselves carry no meaning.


Spend 550.0, orders 3.0
Spend 800.0, orders 24.25
Spend 2425.0, orders 5.0
Spend 2550.0, orders 27.75


<style>
.ch15-lab-grader { max-width: 900px; margin-top: 1rem; }
.ch15-lab-feedback-list { display: grid; gap: 0.45rem; margin: 0 0 1rem; }
.ch15-lab-feedback-list[hidden] { display: none; }
.ch15-lab-feedback-row { border: 1px solid #d8dee4; border-radius: 6px; padding: 0.55rem 0.65rem; }
.ch15-lab-feedback-row strong { margin-right: 0.35rem; }
.ch15-lab-feedback-row.correct { border-color: #aceebb; background: #dafbe1; color: #116329; }
.ch15-lab-feedback-row.incorrect { border-color: #ffcecb; background: #ffebe9; color: #cf222e; }
.ch15-lab-actions { display: flex; gap: 0.75rem; align-items: center; margin: 1rem 0 0; flex-wrap: wrap; }
.ch15-lab-actions button { border: 1px solid #176de8; border-radius: 6px; background: #176de8; color: white; padding: 0.55rem 0.9rem; cursor: pointer; font-weight: 600; }
.ch15-lab-actions button.secondary { background: white; color: #176de8; }
.ch15-lab-actions.is-locked { opacity: 0.55; }
.ch15-lab-actions button:disabled { cursor: not-allowed; opacity: 0.65; }
.ch15-lab-result { font-weight: 700; }
</style>

<div class='ch15-lab-grader' id='ch15-lab-grader'>
  <div class='ch15-lab-feedback-list' aria-live='polite' hidden>
    <div class='ch15-lab-feedback-row' data-question='q1'><strong>Question 1</strong><span></span></div>
    <div class='ch15-lab-feedback-row' data-question='q2'><strong>Question 2</strong><span></span></div>
    <div class='ch15-lab-feedback-row' data-question='q3'><strong>Question 3</strong><span></span></div>
    <div class='ch15-lab-feedback-row' data-question='q4'><strong>Question 4</strong><span></span></div>
    <div class='ch15-lab-feedback-row' data-question='q5'><strong>Question 5</strong><span></span></div>
  </div>
  <div class='ch15-lab-actions'>
    <button type='button' id='ch15-lab-submit' disabled>Submit</button>
    <button type='button' class='secondary' id='ch15-lab-reset'>Clear Results</button>
    <span class='ch15-lab-result' id='ch15-lab-result' aria-live='polite'></span>
  </div>
</div>

<script>
(function () {
  var labId = 'ch15-lab';

  function getParams() {
    var params = new URLSearchParams(window.location.search);
    return {
      canvas_course_id: params.get('canvas_course_id') || params.get('course_id') || '',
      canvas_assignment_id: params.get('canvas_assignment_id') || params.get('assignment_id') || '',
      canvas_user_id: params.get('canvas_user_id') || params.get('user_id') || ''
    };
  }

  function loadSessionIdentity(identity, actions, submit, result) {
    submit.disabled = true;
    actions.classList.add('is-locked');
    result.textContent = 'Sign in from the account menu to submit this assignment.';

    return fetch('/api/v1/session.php', { credentials: 'same-origin' })
      .then(function (response) { return response.ok ? response.json() : null; })
      .then(function (payload) {
        if (!payload || !payload.authenticated || !payload.identity) return identity;
        identity.canvas_user_id = payload.identity.canvas_user_id || identity.canvas_user_id || '';
        identity.student_identifier = payload.identity.student_identifier || '';
        submit.disabled = false;
        actions.classList.remove('is-locked');
        result.textContent = payload.identity.display_name ? 'Signed in as ' + payload.identity.display_name + '.' : 'Signed in.';
        return identity;
      })
      .catch(function () { return identity; });
  }

  function codeFromCell(cell) {
    var cm = cell.querySelector('.CodeMirror');
    if (cm && cm.CodeMirror && typeof cm.CodeMirror.getValue === 'function') return cm.CodeMirror.getValue();
    var cmContent = cell.querySelector('.cm-content');
    if (cmContent && cmContent.cmView && cmContent.cmView.view) return cmContent.cmView.view.state.doc.toString();
    var textarea = cell.querySelector('textarea');
    if (textarea) return textarea.value;
    var pre = cell.querySelector('.cell_input pre');
    return pre ? pre.textContent : '';
  }

  function collectCodeCells() {
    var cells = Array.prototype.slice.call(document.querySelectorAll('div.cell.tag_thebe-interactive')).slice(0, 5);
    var code = {};
    cells.forEach(function (cell, index) {
      code['q' + (index + 1)] = codeFromCell(cell);
    });
    return code;
  }

  function applyFeedback(grader, feedback) {
    var list = grader.querySelector('.ch15-lab-feedback-list');
    list.hidden = false;
    ['q1', 'q2', 'q3', 'q4', 'q5'].forEach(function (question) {
      var row = grader.querySelector('[data-question=' + question + ']');
      var message = row.querySelector('span');
      var item = feedback[question];
      row.className = 'ch15-lab-feedback-row';
      if (!item) return;
      message.textContent = item.message + ' (' + item.score + ' / ' + item.max_score + ')';
      row.classList.add(item.correct ? 'correct' : 'incorrect');
    });
  }

  function clearFeedback(grader) {
    var list = grader.querySelector('.ch15-lab-feedback-list');
    list.hidden = true;
    grader.querySelectorAll('.ch15-lab-feedback-row').forEach(function (row) {
      row.className = 'ch15-lab-feedback-row';
      var message = row.querySelector('span');
      if (message) message.textContent = '';
    });
  }

  function saveAttempt(payload) {
    return fetch('/api/v1/lab-attempts.php', {
      method: 'POST',
      headers: { 'Content-Type': 'application/json' },
      credentials: 'same-origin',
      body: JSON.stringify(payload)
    }).then(function (response) {
      return response.text().then(function (text) {
        var payload = text ? JSON.parse(text) : null;
        if (!payload || !response.ok || !payload.ok) {
          throw new Error((payload && (payload.message || payload.error)) || 'Submission API is unavailable.');
        }
        return payload;
      });
    });
  }

  function initLabGrader() {
    var grader = document.getElementById('ch15-lab-grader');
    if (!grader || grader.dataset.ready === 'true') return;
    grader.dataset.ready = 'true';

    var identity = getParams();
    var actions = grader.querySelector('.ch15-lab-actions');
    var result = document.getElementById('ch15-lab-result');
    var submit = document.getElementById('ch15-lab-submit');
    var reset = document.getElementById('ch15-lab-reset');

    loadSessionIdentity(identity, actions, submit, result);

    submit.addEventListener('click', function () {
      clearFeedback(grader);
      submit.disabled = true;
      result.textContent = 'Submitting code.';

      saveAttempt({
        lab_id: labId,
        code: collectCodeCells(),
        canvas_course_id: identity.canvas_course_id,
        canvas_assignment_id: identity.canvas_assignment_id,
        canvas_user_id: identity.canvas_user_id
      })
        .then(function (payload) {
          var best = (payload.best_score === null || typeof payload.best_score === 'undefined') ? payload.score : payload.best_score;
          var attempt = (payload.attempt_count === null || typeof payload.attempt_count === 'undefined') ? '' : ' Attempt ' + payload.attempt_count + '.';
          applyFeedback(grader, payload.feedback || {});
          result.textContent = 'Submission saved.' + attempt + ' Score: ' + payload.score + ' / ' + payload.max_score + '. Best score: ' + best + ' / ' + payload.max_score + '. Canvas sync: ' + payload.canvas_sync_status + '.';
        })
        .catch(function (error) {
          result.textContent = 'Could not save this attempt: ' + error.message;
        })
        .finally(function () {
          submit.disabled = false;
        });
    });

    reset.addEventListener('click', function () {
      clearFeedback(grader);
      result.textContent = '';
    });
  }

  if (document.readyState === 'loading') {
    document.addEventListener('DOMContentLoaded', initLabGrader);
  } else {
    initLabGrader();
  }
})();
</script>
